# Tugas 3 Komputasi Statistika II

> Nama: Sahda Huwaidah Estiningtyas

> NIM: 24/545080/PA/23156

> email: estiningtyas.leaf@gmail.com

# **Rangkuman Bab 6. Resampling Methods dan Contoh Kasus dengan Python**

## A. Rangkuman Materi Resampling Methods

Resampling Methods merupakan sekumpulan teknik statistik yang bekerja dengan cara melakukan pengambilan sampel ulang dari data yang tersedia untuk memperoleh informasi tambahan mengenai performa model atau kestabilan suatu estimator. Teknik ini sangat penting ketika peneliti tidak memiliki data uji terpisah yang besar, namun tetap ingin mengetahui seberapa baik model akan bekerja pada data baru serta seberapa akurat suatu parameter yang diestimasi.

Menurut pembahasan pada file ISLP dari Praktikum Komputasi Statistika II, dua metode utama dalam resampling adalah **Cross-Validation** dan **Bootstrap**. Cross-validation digunakan untuk menilai kemampuan prediksi model (model assessment) sekaligus membantu memilih kompleksitas model terbaik (model selection), sedangkan bootstrap digunakan untuk mengukur ketidakpastian atau standard error dari suatu estimator.

### 1. Validation Set Approach

Validation Set Approach adalah metode paling sederhana, yaitu membagi data menjadi dua bagian:

* training set untuk membangun model,
* validation set untuk menguji model.

Nilai error pada validation set digunakan sebagai estimasi test error. Kelebihannya adalah mudah dilakukan, tetapi kelemahannya cukup besar karena hasil sangat bergantung pada pembagian data secara acak, serta model hanya dilatih menggunakan sebagian data sehingga performanya bisa kurang optimal.

### 2. Leave-One-Out Cross Validation (LOOCV)

LOOCV membagi data menjadi n bagian, di mana setiap iterasi hanya satu observasi dijadikan data validasi dan sisanya menjadi data training. Proses ini dilakukan sebanyak n kali hingga semua observasi pernah menjadi data validasi.

Kelebihan LOOCV:

* bias kecil karena hampir seluruh data digunakan untuk training,
* hasil deterministik.

Kekurangan LOOCV:

* komputasi berat,
* varians estimasi cenderung tinggi.

### 3. k-Fold Cross Validation

Pada metode ini data dibagi menjadi k kelompok (fold) dengan ukuran relatif sama. Setiap fold bergantian menjadi data validasi, sedangkan fold lainnya menjadi training set. Nilai error akhir diperoleh dari rata-rata seluruh fold.

Umumnya digunakan:

* 5-Fold CV
* 10-Fold CV

Metode ini dianggap paling seimbang karena memiliki trade-off yang baik antara bias, varians, dan efisiensi komputasi.

### 4. Bias-Variance Trade-Off pada Cross Validation

* Semakin besar k (misalnya LOOCV), bias makin kecil tetapi varians makin tinggi.
* Semakin kecil k, varians lebih kecil tetapi bias meningkat.

Karena itu, k = 5 atau k = 10 sering menjadi pilihan praktis terbaik.

### 5. Bootstrap

Bootstrap adalah metode pengambilan sampel ulang sebanyak B kali dari data asli dengan teknik **sampling with replacement**. Artinya satu observasi dapat terambil lebih dari satu kali dalam satu sampel bootstrap.

Dari setiap sampel bootstrap dihitung estimator yang diinginkan, lalu variasi seluruh estimator tersebut digunakan untuk menghitung:

* standard error,
* confidence interval,
* kestabilan parameter.


## B. Contoh Kasus Baru 1
### **Menentukan Derajat Polynomial Regression Terbaik dengan 10-Fold Cross Validation**

**Soal:**

Sebuah perusahaan ingin memprediksi penjualan es krim harian berdasarkan suhu udara. Diduga hubungan suhu dengan penjualan tidak linear sempurna karena pada suhu terlalu ekstrem penjualan dapat melandai.

Data yang tersedia berjumlah 60 hari.

Tujuan analisis adalah menentukan apakah model bersifat linear, quadratic, atau cubic.

In [ ]:
#. Import Library
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline

In [ ]:
# Membuat Data Simulasi
np.random.seed(10)
suhu = np.random.uniform(20, 38, 60)
penjualan = 5 + 2.3*suhu - 0.03*(suhu**2) + np.random.normal(0, 1.5, 60)

data = pd.DataFrame({'Suhu': suhu, 'Penjualan': penjualan})
data.head()

,Suhu,Penjualan
0,33.883772,49.752712
1,20.373535,39.712074
2,31.405668,51.235613
3,33.478470,49.752431
4,28.973126,46.286520


In [ ]:
# Menentukan 10-Fold CV
X = data[['Suhu']]
y = data['Penjualan']
kf = KFold(n_splits=10, shuffle=True, random_state=1)

#. Menghitung CV Error Setiap Derajat Polynomial
cv_error = []

for d in range(1,4):
    model = make_pipeline(PolynomialFeatures(d), LinearRegression())
    mse = -cross_val_score(model, X, y,
                           cv=kf,
                           scoring='neg_mean_squared_error').mean()
    cv_error.append(mse)
    print(f'Degree {d}: CV MSE = {mse:.4f}')

Degree 1: CV MSE = 2.9791
Degree 2: CV MSE = 2.6499
Degree 3: CV MSE = 2.5920


**Interpretasi:**

```python
Degree 1: CV MSE = 2.9791
Degree 2: CV MSE = 2.6499
Degree 3: CV MSE = 2.5920
```

Berdasarkan output, model degree 3 menghasilkan error terkecil. Ini menunjukkan hubungan suhu dan penjualan cukup direpresentasikan oleh kurva kubik. Dapat disimpulkan bahwa model polynomial cubic adalah model paling optimal untuk prediksi penjualan es krim karena memiliki estimasi test error paling kecil berdasarkan 10-Fold Cross Validation.

## C. Contoh Kasus Baru 2
### **Mengestimasi Standard Error Median Pendapatan UMKM dengan Bootstrap**

**Soal:**

Seorang peneliti ingin mengetahui kestabilan nilai median pendapatan bulanan UMKM. Karena distribusi pendapatan biasanya tidak normal dan cenderung skewed, standard error median sulit dihitung dengan rumus biasa. Oleh karena itu, digunakan bootstrap.

In [ ]:
# Data Simulasi Pendapatan UMKM
np.random.seed(20)
pendapatan = np.random.gamma(shape=4, scale=700, size=80)

umkm = pd.DataFrame({'Pendapatan': pendapatan})
umkm.head()

,Pendapatan
0,3943.080528
1,2838.256867
2,3076.354244
3,532.762517
4,4043.096631


In [ ]:
# Median Data Asli
median_asli = np.median(umkm['Pendapatan'])
median_asli

np.float64(2874.284112203828)

In [ ]:
# Proses Bootstrap
B = 1000
bootstrap_median = []

for i in range(B):
    sample = umkm['Pendapatan'].sample(n=len(umkm), replace=True)
    bootstrap_median.append(np.median(sample))

# Menghitung Bootstrap Standard Error
bootstrap_se = np.std(bootstrap_median)
print('Median Asli :', median_asli)
print('Bootstrap SE:', bootstrap_se)

Median Asli : 2874.284112203828
Bootstrap SE: 249.75003013576364


**Interpretasi:**

Berdasarkan output, estimasi median pendapatan UMKM sebesar 2874,28 ribu rupiah memiliki ketidakpastian sekitar 249,75. Nilai ini menunjukkan bahwa jika pengambilan sampel diulang berkali-kali, median pendapatan dapat berfluktuasi sekitar angka tersebut. Dapat disimpulkan bahwa bootstrap berhasil memberikan ukuran ketelitian estimator median tanpa memerlukan asumsi distribusi normal.


## D. Perbandingan Singkat Kapan Menggunakan Masing-Masing

| Metode         | Tujuan Utama                      | Cocok Digunakan Saat        |
| -------------- | --------------------------------- | --------------------------- |
| Validation Set | Estimasi cepat test error         | Data cukup besar            |
| LOOCV          | Estimasi test error bias kecil    | Data kecil                  |
| k-Fold CV      | Model selection paling stabil     | Hampir semua kasus prediksi |
| Bootstrap      | Estimasi standard error estimator | Ketidakpastian parameter    |

## E. Kesimpulan Umum

Resampling Methods menjadi alat penting dalam statistika modern karena memungkinkan peneliti mengevaluasi model dan estimator tanpa harus memiliki data tambahan. Cross-validation berperan besar dalam memilih model prediksi terbaik melalui estimasi test error, sedangkan bootstrap sangat efektif untuk mengukur akurasi estimator ketika pendekatan teoritis sulit dilakukan. Melalui dua contoh kasus baru di atas, terlihat bahwa Resampling Methods tidak hanya relevan untuk data akademik seperti pada modul, tetapi juga sangat aplikatif pada kasus bisnis dan ekonomi nyata.